# Análise exploratória dos dados de câncer no Brasil

Dados provenientes do INCA (Instituto Nacional de Câncer) foram carregados para análise. Fonte dos dados: [INCA - Registro Hospitalar de Câncer](https://irhc.inca.gov.br/RHCNet/visualizaTabNetExterno.action) e [Kaggle](https://www.kaggle.com/dfsets/rafatrindade/onco-360?select=raw_inca_registro_hospitalar.parquet).

In [0]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import re

In [0]:
df_spark = spark.read.parquet('/Volumes/workspace/default/cancer_data/raw_inca_registro_hospitalar.parquet')
display(df_spark)

In [0]:
parquet_file_path = '/Volumes/workspace/default/cancer_data/raw_inca_registro_hospitalar.parquet' 
df = pd.read_parquet(parquet_file_path, engine='pyarrow')

In [0]:
df.info()

#### Colunas deletadas

- VALOR_TOT: todos tem o mesmo valor
- DTTRIAGE: irrelevante em comparação com outras datas e tem muitos valores nan
- MUUH: decidimos deixar apenas a UF
- CNES: como dados do hospital deixamos apenas a UF
- DTINITRT: ja tem outra feature semelhante
- LOCTUPRO: 99% de valores nulos
- LATERALI: como n temos a localização do tumor (loctupro) acreditamos que essa feature n agrega valor
- DTPRICON: ja tem uma feature com a data completa, essa é só o ano e além disso tem mais valores nan
- ANTRI: ja tem uma feature com a data completa, essa é só o ano e além disso tem mais valores nan
- ESTCONJ: irrelevante para o modelo
- PROCEDEN: acreditamos que a localização do hospital já cobre esse aspecto
- CLITRAT: irrelevante, pouco contato com o paciente, hospital tende a ser mais importante
- CLIATEN: irrelevante, pouco contato com o paciente, hospital tende a ser mais importante
- LOCALNAS: acreditamos ser mais importante a localização do hospital do que a do nascimento da pessoa, visto que ela pode ter se mudado e nem ter mais relação com a cidade natal
- ESTADIAG: muitos valores nan
- ESTADIAM: muitos valores nan
- ANOPRIDI: tem DTDIAGNO que é mais completa
- OUTROESTA: falta de informação
- LOCTUDET: LOCTUPRI tem a mesma informação e de forma mais completa

In [0]:
df = df.drop(columns=['VALOR_TOT', 'DTTRIAGE', 'MUUH', 'CNES', 'DTINITRT', 'LOCTUPRO', 'LATERALI', 'DTPRICON', 'ANTRI', 'ESTCONJ', 'PROCEDEN', 'CLITRAT', 'CLIATEN', 'LOCALNAS', 'ESTADIAG', 'ESTADIAM', 'ANOPRIDI', 'OUTROESTA', 'LOCTUDET'])

In [0]:
df = df.replace('nan', np.nan)

In [0]:
missing_values = df.isnull().sum()
missing_values = missing_values[missing_values > 0].sort_values(ascending=False)
plt.figure(figsize=(12, 6))
sns.barplot(x=missing_values.index, y=missing_values.values, palette='viridis')
plt.xticks(rotation=90)
plt.title('Número de Valores Faltantes por Coluna')
plt.ylabel('Número de Valores Faltantes')
plt.xlabel('Colunas')
plt.tight_layout()
plt.show()

In [0]:
len(df)

In [0]:
df = df.dropna(subset=[col for col in df.columns if col != 'DATAOBITO'])

In [0]:
len(df)

In [0]:
#df2 = df

In [0]:
#df_spark = spark.createDataFrame(df)
#df_spark.write.saveAsTable("after_nan_table_without_ptnm")

## Coerência dos dados

In [0]:
#df = spark.table("after_nan_table_without_ptnm").toPandas()

In [0]:
# Convertendo colunas de datas para o formato datetime
date_cols = ['DTDIAGNO', 'DATAINITRT', 'DATAOBITO', 'DATAPRICON']

for col in date_cols:
    if col in df.columns:
        df[col] = pd.to_datetime(
            df[col],
            format='%d/%m/%Y',
            errors='coerce'
        )

In [0]:
# Investigando as colunas de data
print("DTDIAGNO (Data de Diagnóstico):")
print(f"  Min: {df['DTDIAGNO'].min()}")
print(f"  Max: {df['DTDIAGNO'].max()}")
print(f"  NaN: {df['DTDIAGNO'].isna().sum()}")

print("\nDATAINITRT (Data de Início de Tratamento):")
print(f"  Min: {df['DATAINITRT'].min()}")
print(f"  Max: {df['DATAINITRT'].max()}")
print(f"  NaN: {df['DATAINITRT'].isna().sum()}")

print("\nDATAOBITO (Data de Óbito):")
print(f"  Min: {df['DATAOBITO'].min()}")
print(f"  Max: {df['DATAOBITO'].max()}")
print(f"  NaN: {df['DATAOBITO'].isna().sum()}")

print("\nDATAPRICON (Data Primeira Consulta):")
print(f"  Min: {df['DATAPRICON'].min()}")
print(f"  Max: {df['DATAPRICON'].max()}")
print(f"  NaN: {df['DATAPRICON'].isna().sum()}")

In [0]:
#Excluindo datas problemáticas
print("Datas de diagnóstico problemáticas (< 1900 ou > 2023):")
diagnostico_invalido = df[(df['DTDIAGNO'] < '1900-01-01') | (df['DTDIAGNO'] > '2023-12-31')]
print(f"   Total: {len(diagnostico_invalido)} casos ({len(diagnostico_invalido)/len(df)*100:.2f}%)")
df = df.drop(diagnostico_invalido.index)

print("Datas de tratamento problemáticas (< 1900 ou > 2023):")
tratamento_invalido = df[(df['DATAINITRT'] < '1900-01-01') | (df['DATAINITRT'] > '2023-12-31')] 
print(f"   Total: {len(tratamento_invalido)} casos ({len(tratamento_invalido)/len(df)*100:.2f}%)")
df = df.drop(tratamento_invalido.index) 

print("Datas de óbito problemáticas (< 1900 ou > 2023):")
obito_invalido = df[(df['DATAOBITO'] < '1900-01-01') | (df['DATAOBITO'] > '2023-12-31')]
print(f"   Total: {len(obito_invalido)} casos ({len(obito_invalido)/len(df)*100:.2f}%)")
df = df.drop(obito_invalido.index) 

print("\nTratamento ANTES do diagnóstico:")
tratamento_antes = df[(df['DATAINITRT'].notna()) & (df['DTDIAGNO'].notna()) & (df['DATAINITRT'] < df['DTDIAGNO'])]
print(f"   Total: {len(tratamento_antes)} casos ({len(tratamento_antes)/len(df)*100:.2f}%)")
df = df.drop(tratamento_antes.index) 

print("\nÓbito ANTES do diagnóstico:")
obito_antes_diag = df[(df['DATAOBITO'].notna()) & (df['DTDIAGNO'].notna()) & (df['DATAOBITO'] < df['DTDIAGNO'])]
print(f"   Total: {len(obito_antes_diag)} casos ({len(obito_antes_diag)/len(df)*100:.2f}%)")
df = df.drop(obito_antes_diag.index) 

print("\nÓbito ANTES do tratamento:")
obito_antes_trat = df[(df['DATAOBITO'].notna()) & (df['DATAINITRT'].notna()) & (df['DATAOBITO'] < df['DATAINITRT'])]
print(f"   Total: {len(obito_antes_trat)} casos ({len(obito_antes_trat)/len(df)*100:.2f}%)")
df = df.drop(obito_antes_trat.index) 

print("\nDatas problemáticas deletadas com sucesso.")

In [0]:
# Excluindo datas null (vindas do errors=coerce), exceto para DATAOBITO em que null pode significar que a pessoa esta viva (ent precisamos dessa info)
df = df.dropna(subset=['DTDIAGNO', 'DATAINITRT'])
df.isnull().sum()

In [0]:
# Limpeza e tratamento da coluna IDADE como inteiro
df['IDADE'] = pd.to_numeric(df['IDADE'], errors='coerce')
df = df[(df['IDADE'].notna()) & (df['IDADE'] >= 0) & (df['IDADE'] <= 100)]
df['IDADE'] = df['IDADE'].astype(int)

In [0]:
# Investigando valores estranhos de TNM
tnm_series = df['TNM'].dropna().astype(str)
count_with_letters = tnm_series[tnm_series.str.contains(r'[A-Za-z]', regex=True)].count()
count_999 = tnm_series[tnm_series == '999'].count()
count_888 = tnm_series[tnm_series == '888'].count()
count_less_3 = tnm_series[tnm_series.str.len() < 3].count()
count_more_3 = tnm_series[tnm_series.str.len() > 3].count()

print(f"Total TNM não nulos: {len(tnm_series)}")
print(f"Com letras: {count_with_letters}")
print(f"Com valor '999': {count_999}")
print(f"Com valor '888': {count_888}")
print(f"Com menos de 3 caracteres: {count_less_3}")
print(f"Com mais de 3 caracteres: {count_more_3}")

print("Casos com letras:")
print(tnm_series[tnm_series.str.contains(r'[A-Za-z]', regex=True)].unique())

In [0]:
#FIXME: considerar tbm a letra 'X' ('nao pode ser avaliado')??
#FIXME: talvez primeiro separar T, N, M e depois julgar separadamente se esta valido ou nao, ao inves de colocar como nulo quando apenas 1 esta incorreto

# Deixando apenas valores de TNM validos
def clean_tnm(value):
    if pd.isna(value):
        return np.nan
    
    value = str(value).strip()
    # T: 0-4, N: 0-3, M: 0-1
    if re.fullmatch(r'[0-4][0-3][0-1]', value):
        return value
    
    return np.nan


df['TNM'] = df['TNM'].apply(clean_tnm)
df['PTNM'] = df['PTNM'].apply(clean_tnm)

In [0]:
len(df)

In [0]:
# Excluindo linhas que contenham alguma feature com valor fora do dominio valido (https://raw.githubusercontent.com/rafa-trindade/oncoped-360/main/docs/dicionario_inca_registro_hospitalar.pdf)
validos = {
    "TPCASO": {"1", "2"},
    "SEXO": {"1", "2"},
    "RACACOR": {"1", "2", "3", "4", "5"},
    "INSTRUC": {"1", "2", "3", "4", "5", "6"},
    "HISTFAMC": {"1", "2"},
    "ALCOOLIS": {"1", "2", "3", "4", "8"},
    "TABAGISM": {"1", "2", "3", "4", "8"},
    "ORIENC": {"1", "2", "3", "8"},
    "EXDIAG": {"1", "2", "3", "4", "5", "8"},
    "DIAGANT": {"1", "2", "3", "4"},
    "BASMAIMP": {"1", "2", "3", "4", "5", "6", "7"},
    "MAISUMTU": {"1", "2", "3"},
    "RZNTR": {"1", "2", "3", "4", "5", "6", "7", "8"},
    "PRITRATH": {"1", "2", "3", "4", "5", "6", "7", "8"},
    "ESTDFIMT": {"1", "2", "3", "4", "5", "6", "8"},
    "BASDIAGSP": {"1", "2", "3"},
}

ufs_validas = {
    "AC","AL","AP","AM","BA","CE","DF","ES","GO","MA","MT","MS",
    "MG","PA","PB","PR","PE","PI","RJ","RN","RS","RO","RR","SC",
    "SP","SE","TO"
}

mask_valido = pd.Series(True, index=df.index)

for col, valores in validos.items():
    if col in df.columns:
        mask_valido &= df[col].isin(valores)

for col in ["ESTADRES", "UFUH"]:
    if col in df.columns:
        mask_valido &= df[col].isin(ufs_validas)

if "OCUPACAO" in df.columns:
    mask_valido &= df["OCUPACAO"].str.fullmatch(r"[1-9]\d*") #FIXME: considerar 'ocupaçao ignorada' (mais de tres 9)? desse jeito esta considerando

if "TIPOHIST" in df.columns:
    mask_valido &= df["TIPOHIST"].str.fullmatch(r"(8|9)\d{3}/[012369]")

if "LOCTUPRI" in df.columns:
    mask_valido &= df["LOCTUPRI"].str.fullmatch(r"C(0[0-9]|[1-7][0-9]|80)\.[0-9]")

# remove linhas invalidas
df = df.loc[mask_valido].reset_index(drop=True)

In [0]:
len(df)

In [0]:
df.isnull().sum()

In [0]:
tnm_nulls_by_date = df[df['TNM'].isna()].groupby(df['DTDIAGNO'].dt.year).size()

plt.figure(figsize=(10, 5))
sns.lineplot(x=tnm_nulls_by_date.index, y=tnm_nulls_by_date.values, marker='o')
plt.title('Valores TNM Nulos ao Longo do Tempo (por Ano de Diagnóstico)')
plt.xlabel('Ano de Diagnóstico')
plt.ylabel('Quantidade de TNM Nulos (log)')
plt.yscale('log')
plt.tight_layout()
plt.show()

In [0]:
tnm_nulls_by_date = df[df['PTNM'].isna()].groupby(df['DTDIAGNO'].dt.year).size()

plt.figure(figsize=(10, 5))
sns.lineplot(x=tnm_nulls_by_date.index, y=tnm_nulls_by_date.values, marker='o')
plt.title('Valores PTNM Nulos ao Longo do Tempo (por Ano de Diagnóstico)')
plt.xlabel('Ano de Diagnóstico')
plt.ylabel('Quantidade de PTNM Nulos (log)')
plt.yscale('log')
plt.tight_layout()
plt.show()

## Criando novas features

- separar idade em faixas etárias?
- separar datas em anos (ou em periodos?), manter meses e dias?

In [0]:
df['STATUS_VITAL'] = np.where(
    (df['ESTDFIMT'] == '6') | ((df['DATAOBITO'].notna())),
    0,
    1
)

# se a pessoa estiver viva: data final do dataset (dez/2023) - data do diagnóstico
# se estiver morta: data de óbito - data do diagnóstico
data_final = pd.to_datetime('2023-12-31')
df['TEMPO_SEGUIMENTO'] = np.where(
    df['STATUS_VITAL'] == 1,
    (data_final - df['DTDIAGNO']).dt.days,
    (df['DATAOBITO'] - df['DTDIAGNO']).dt.days
)

df['TEMPO_ATE_TRATAMENTO'] = (df['DATAINITRT'] - df['DTDIAGNO']).dt.days

In [0]:
#FIXME: essa proporção nao esta muito discrepante para um algoritmo de ml?
status_counts = df['STATUS_VITAL'].value_counts(normalize=True)
plt.figure(figsize=(6, 4))
sns.barplot(x=['Óbito' if i == 0 else 'Vivo' for i in status_counts.index], y=status_counts.values, palette='viridis')
plt.title('Gráfico STATUS_VITAL')
plt.ylabel('Proporção')
plt.xlabel('Status Vital')
plt.ylim(0, 1)
plt.tight_layout()
plt.show()

In [0]:
# Decodificar T, N, M
df['T_cat'] = df['TNM'].str[0]
df['N_cat'] = df['TNM'].str[1]
df['M_cat'] = df['TNM'].str[2]

df = df.drop(columns=['TNM'])

In [0]:
#TODO: Decodificar outras variaveis relevantes, como loctupri?

In [0]:
#TODO: excluir features de datas desnecessarias, se vamos trabalhar com as variaveis de tempo, entao podemos excluir as datas, se n fica redundante

## Encode

Fazer one-hot encoding das variáveis categóricas selecionadas, como tnm, loctupri, sexo, etc

Para um modelo de rsf, os códigos de CID-O não devem ser usados como valores numéricos brutos, pois isso introduz uma ordem artificial sem significado clínico; em vez disso:

- tratar a variável como categórica, preferencialmente usando a LOCTUPRI (CID-O, 4 dígitos) por ser mais informativa. Caso o número de observações seja limitado, usar LOCTUDET (CID-O, 3 dígitos)
- agrupar categorias raras (por exemplo, localizações com baixa frequência) em uma classe “Outros”
- aplicar one-hot encoding 

In [0]:
#TODO : coisas acima para a feature de CID-O

In [0]:
#TODO: fazer encoding de todas outras variaveis necessarias

## Gráficos

In [0]:
map_sexo = {
    '1': 'Masculino',
    '2': 'Feminino'
}

map_tabagismo = {
    '1': 'Nunca',
    '2': 'Ex-consumidor',
    '3': 'Sim',
    '4': 'Não avaliado',
    '8': 'Não se aplica',
    '9': 'Sem informação'
}

map_alcool = map_tabagismo.copy()

map_racacor = {
    '1': 'Branca',
    '2': 'Preta',
    '3': 'Amarela',
    '4': 'Parda',
    '5': 'Indígena',
    '9': 'Sem informação'
}

map_estdfimt = {
    '1': 'Remissão completa',
    '2': 'Remissão parcial',
    '3': 'Doença estável',
    '4': 'Progressão',
    '5': 'Suporte terapêutico',
    '6': 'Óbito',
    '8': 'Não se aplica',
    '9': 'Sem informação'
}

map_pritrath = {
    '1': 'Nenhum',
    '2': 'Cirurgia',
    '3': 'Radioterapia',
    '4': 'Quimioterapia',
    '5': 'Hormonioterapia',
    '6': 'Transplante de medula óssea',
    '7': 'Imunoterapia',
    '8': 'Outros',
    '9': 'Sem informação'
}

map_basmaimp = {
    '1': 'Clínica',
    '2': 'Pesquisa clínica',
    '3': 'Exame por imagem',
    '4': 'Marcadores tumorais',
    '5': 'Citologia',
    '6': 'Histologia da metástase',
    '7': 'Histologia do tumor primário',
    '9': 'Sem informação'
}

map_exdiag = {
    '1': 'Exame clínico / Patologia clínica',
    '2': 'Exames por imagem',
    '3': 'Endoscopia / Cirurgia exploradora',
    '4': 'Anatomia patológica',
    '5': 'Marcadores tumorais',
    '8': 'Não se aplica',
    '9': 'Sem informação'
}

map_orienc = {
    '1': 'SUS',
    '2': 'Não SUS',
    '3': 'Conta própria',
    '8': 'Não se aplica',
    '9': 'Sem informação'
}

map_tp_caso = {
    '1': 'Analítico',
    '2': 'Não analítico',
}

map_status_vital = {
    0: 'Óbito',
    1: 'Vivo'
}

In [0]:
df['DTDIAGNO_Year'] = pd.to_datetime(df['DTDIAGNO'], errors='coerce').dt.year
obito_counts = df[df['DATAOBITO'].isna()].groupby('DTDIAGNO_Year').size()
plt.figure(figsize=(10, 6))
sns.barplot(x=obito_counts.index, y=obito_counts.values, palette='viridis')
plt.title("Count of nan in DATAOBITO by Year of DTDIAGNO")
plt.xlabel("Year of DTDIAGNO")
plt.ylabel("Count of nan in DATAOBITO")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [0]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

df['SEXO'].map(map_sexo).value_counts().plot(kind='bar', ax=axes[0])
axes[0].set_title('Distribuição por Sexo')
axes[0].set_ylabel('Pacientes')

df['RACACOR'].map(map_racacor).value_counts().plot(kind='bar', ax=axes[1])
axes[1].set_title('Distribuição por Raça/Cor')

df['IDADE'].dropna().plot(kind='hist', bins=20, ax=axes[2])
axes[2].set_title('Distribuição de Idade')
axes[2].set_xlabel('Idade')

plt.tight_layout()
plt.show()

In [0]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df['TABAGISM'].map(map_tabagismo).value_counts().plot(kind='bar', ax=axes[0])
axes[0].set_title('Histórico de Tabagismo')

df['ALCOOLIS'].map(map_alcool).value_counts().plot(kind='bar', ax=axes[1])
axes[1].set_title('Histórico de Alcoolismo')

plt.tight_layout()
plt.show()

In [0]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

df['BASMAIMP'].map(map_basmaimp).value_counts().plot(kind='bar', ax=axes[0])
axes[0].set_title('Base Mais Importante do Diagnóstico')

df['EXDIAG'].astype(str).str.strip().map(map_exdiag).fillna('Sem informação') \
    .value_counts().plot(kind='bar', ax=axes[1])
axes[1].set_title('Exames para Diagnóstico')

plt.tight_layout()
plt.show()

In [0]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

df['PRITRATH'].map(map_pritrath).value_counts().plot(kind='bar', ax=axes[0])
axes[0].set_title('Primeiro Tratamento Recebido')

df['ESTDFIMT'].map(map_estdfimt).value_counts().plot(kind='bar', ax=axes[1])
axes[1].set_title('Estado da Doença ao Final do Tratamento')

plt.tight_layout()
plt.show()

In [0]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

df.loc[df['STATUS_VITAL'] == 0, 'IDADE'].dropna().plot(
    kind='hist', bins=20, alpha=0.7, ax=axes[0]
)
axes[0].set_title('Idade - Não Óbito')
axes[0].set_xlabel('Idade')

df.loc[df['STATUS_VITAL'] == 1, 'IDADE'].dropna().plot(
    kind='hist', bins=20, alpha=0.7, ax=axes[1]
)
axes[1].set_title('Idade - Óbito')
axes[1].set_xlabel('Idade')

plt.tight_layout()
plt.show()

In [0]:
tpcaso = (
    df['TPCASO']
    .astype(str)
    .str.strip()
    .map(map_tp_caso)
    .fillna('Sem informação')
)

cross = pd.crosstab(tpcaso, df['STATUS_VITAL'].map(map_status_vital))
cross_prop = cross.div(cross.sum(axis=1), axis=0)

cross_prop.plot(
    kind='bar',
    stacked=True,
    figsize=(8, 5)
)

plt.title('Proporção de Óbito por Tipo de Caso')
plt.ylabel('Proporção')
plt.xlabel('Tipo de Caso')
plt.legend(title='Status Vital')
plt.tight_layout()
plt.show()